In [ ]:
"""
Per-Family Error Breakdown for Stacking Ensemble Tc Prediction
==============================================================
Computes R², RMSE, and MAE on the held-out test set, broken down by
superconductor family. Also generates a publication-quality figure.

Input files required (from your pipeline):
  - Final_dataset_83.csv      : full 21,263-row dataset (for family labels)
  - top30_train_83.csv        : training split (30 features + critical_temp)
  - top30_test_83.csv         : test split (30 features + critical_temp)
  - models_83/tuning_50_HPO/  : saved base learner .pkl files

The test split is recreated with the same random_state=42 used in
Section 2.5 to recover family labels for each test row.
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import joblib

from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import StackingRegressor
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor
from sklearn.linear_model import Ridge

# =============================================================================
# 1. LOAD DATA
# =============================================================================
df_full  = pd.read_csv("Final_dataset_83.csv")
df_train = pd.read_csv("top30_vF_xi_Hc_train_83.csv")
df_test  = pd.read_csv("top30_vF_xi_Hc_test_83.csv")

TARGET = "critical_temp"

X_train = df_train.drop(columns=[TARGET])
y_train = df_train[TARGET]
X_test  = df_test.drop(columns=[TARGET])
y_test  = df_test[TARGET]

print(f"Train : {X_train.shape}  |  Test : {X_test.shape}")

# =============================================================================
# 2. RECOVER FAMILY LABELS FOR TEST ROWS
#    Recreate the exact 80/20 split used in Section 2.5 (same random_state)
#    to align family labels with the test partition.
# =============================================================================
# The feature selection step dropped many columns before splitting,
# so we use critical_temp as the join key — it is unique enough per row
# when combined with the fixed random_state split to recover the right rows.
_, full_test_idx = train_test_split(
    df_full.index,
    test_size=0.20,
    random_state=42
)
family_test = df_full.loc[full_test_idx, "family"].reset_index(drop=True)

print(f"\nTest-set family distribution:")
print(family_test.value_counts().to_string())

# =============================================================================
# 3. SCALE FEATURES
# =============================================================================
scaler       = RobustScaler()
X_train_sc   = scaler.fit_transform(X_train)
X_test_sc    = scaler.transform(X_test)

# =============================================================================
# 4. LOAD BASE LEARNERS AND ASSEMBLE STACKING MODEL
# =============================================================================
MODEL_DIR = "models_83/tuning_50_HPO"

lasso_best = joblib.load(f"{MODEL_DIR}/lasso_best.pkl")
knn_best   = joblib.load(f"{MODEL_DIR}/knn_best.pkl")
svr_best   = joblib.load(f"{MODEL_DIR}/svr_best.pkl")
mlp_best   = joblib.load(f"{MODEL_DIR}/mlp_best.pkl")
best_rf    = joblib.load(f"{MODEL_DIR}/best_rf.pkl")
best_lgb   = joblib.load(f"{MODEL_DIR}/best_lgb.pkl")

base_models = [
    ("knn",  knn_best),
    ("svr",  svr_best),
    ("mlp",  mlp_best),
    ("lgbm", best_lgb),
    ("rf",   best_rf),
]

# meta_xgb = XGBRegressor(
#     n_estimators     = 300,
#     learning_rate    = 0.05,
#     max_depth        = 4,
#     colsample_bytree = 0.6,
#     subsample        = 0.6,
#     min_child_weight = 5,
#     gamma            = 0.1,
#     reg_lambda       = 10.0,
#     reg_alpha        = 0.5,
#     random_state     = 42,
#     verbosity        = 0,
#     n_jobs           = -1,
# )

stack = StackingRegressor(
    estimators     = base_models,
    final_estimator= Ridge(alpha=2.0, random_state=42),
    cv             = KFold(n_splits=5, shuffle=True, random_state=42),
    passthrough    = True,
    n_jobs         = -1,
)

print("\nFitting stacking ensemble on training set...")
stack.fit(X_train_sc, y_train)

# =============================================================================
# 5. GENERATE PREDICTIONS AND COMPUTE OVERALL METRICS
# =============================================================================
y_pred = stack.predict(X_test_sc)

overall_r2   = r2_score(y_test, y_pred)
overall_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
overall_mae  = mean_absolute_error(y_test, y_pred)

print(f"\n{'='*55}")
print(f"OVERALL TEST-SET PERFORMANCE")
print(f"  R²   : {overall_r2:.4f}")
print(f"  RMSE : {overall_rmse:.4f} K")
print(f"  MAE  : {overall_mae:.4f} K")
print(f"{'='*55}")

# =============================================================================
# 6. PER-FAMILY BREAKDOWN
# =============================================================================
results = pd.DataFrame({
    "y_true" : y_test.values,
    "y_pred" : y_pred,
    "family" : family_test.values,
})

FAMILY_ORDER = [
    "Cuprates",
    "Iron-based",
    "Conventional / Alloys",
    "Heavy Fermion",
    "Mg-B related",
    "Chalcogenide",
]

PALETTE = {
    "Cuprates"             : "#e41a1c",
    "Iron-based"           : "#377eb8",
    "Conventional / Alloys": "#4daf4a",
    "Heavy Fermion"        : "#984ea3",
    "Mg-B related"         : "#ff7f00",
    "Chalcogenide"         : "#a65628",
}

records = []
print(f"\n{'Family':<28}{'n':>6}{'R²':>10}{'RMSE (K)':>12}{'MAE (K)':>12}")
print("-" * 70)

for fam in FAMILY_ORDER:
    mask = results["family"] == fam
    if mask.sum() < 2:
        continue
    yt = results.loc[mask, "y_true"].values
    yp = results.loc[mask, "y_pred"].values
    r2   = r2_score(yt, yp)
    rmse = np.sqrt(mean_squared_error(yt, yp))
    mae  = mean_absolute_error(yt, yp)
    n    = mask.sum()
    records.append({"family": fam, "n": n,
                    "R2": r2, "RMSE": rmse, "MAE": mae})
    print(f"  {fam:<26}{n:>6}{r2:>10.4f}{rmse:>12.4f}{mae:>12.4f}")

print("-" * 70)
print(f"  {'OVERALL':<26}{len(y_test):>6}"
      f"{overall_r2:>10.4f}{overall_rmse:>12.4f}{overall_mae:>12.4f}")

df_results = pd.DataFrame(records)


In [ ]:

# =============================================================================
# 7. PLOTS
# =============================================================================
fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.38)

# # ── (a) Per-family R² bar chart ──────────────────────────────────────────
# ax_r2 = fig.add_subplot(gs[0, 0])
# colors = [PALETTE[f] for f in df_results["family"]]
# bars = ax_r2.barh(df_results["family"][::-1], df_results["R2"][::-1],
#                    color=colors[::-1], edgecolor="white", linewidth=0.4)
# ax_r2.axvline(overall_r2, color="black", lw=1.2, ls="--",
#                label=f"Overall R² = {overall_r2:.4f}")
# ax_r2.set_xlabel("R²", fontsize=11)
# ax_r2.set_title("(a) Per-Family R²", fontsize=11)
# ax_r2.set_xlim(0, 1.05)
# ax_r2.legend(fontsize=8)
# ax_r2.grid(True, alpha=0.3, axis="x")

# # ── (b) Per-family RMSE bar chart ────────────────────────────────────────
# ax_rm = fig.add_subplot(gs[0, 1])
# ax_rm.barh(df_results["family"][::-1], df_results["RMSE"][::-1],
#             color=colors[::-1], edgecolor="white", linewidth=0.4)
# ax_rm.axvline(overall_rmse, color="black", lw=1.2, ls="--",
#                label=f"Overall RMSE = {overall_rmse:.2f} K")
# ax_rm.set_xlabel("RMSE (K)", fontsize=11)
# ax_rm.set_title("(b) Per-Family RMSE", fontsize=11)
# ax_rm.legend(fontsize=8)
# ax_rm.grid(True, alpha=0.3, axis="x")

# # ── (c) Per-family MAE bar chart ─────────────────────────────────────────
# ax_ma = fig.add_subplot(gs[0, 2])
# ax_ma.barh(df_results["family"][::-1], df_results["MAE"][::-1],
#             color=colors[::-1], edgecolor="white", linewidth=0.4)
# ax_ma.axvline(overall_mae, color="black", lw=1.2, ls="--",
#                label=f"Overall MAE = {overall_mae:.2f} K")
# ax_ma.set_xlabel("MAE (K)", fontsize=11)
# ax_ma.set_title("(c) Per-Family MAE", fontsize=11)
# ax_ma.legend(fontsize=8)
# ax_ma.grid(True, alpha=0.3, axis="x")


In [31]:

fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.28, wspace=0.18)

# ── (d-i) Per-family predicted vs actual scatter ─────────────────────────
scatter_positions = [
    gs[0, 0], gs[0, 1], gs[0, 2],
    gs[1, 0], gs[1, 1], gs[1, 2],
]

for ax_pos, fam in zip(scatter_positions, FAMILY_ORDER):
    ax = fig.add_subplot(ax_pos)
    mask = results["family"] == fam
    yt = results.loc[mask, "y_true"].values
    yp = results.loc[mask, "y_pred"].values
    n  = mask.sum()

    if n < 2:
        ax.text(0.5, 0.5, f"{fam}\n(n < 2)", ha="center", va="center",
                transform=ax.transAxes, fontsize=9)
        ax.set_title(f"{fam}", fontsize=9)
        continue

    r2   = r2_score(yt, yp)
    rmse = np.sqrt(mean_squared_error(yt, yp))

    ax.scatter(yt, yp, color=PALETTE[fam], alpha=0.45, s=10,
               edgecolors="none", rasterized=True)

    lim_min = min(yt.min(), yp.min()) * 0.95
    lim_max = max(yt.max(), yp.max()) * 1.05
    ax.plot([lim_min, lim_max], [lim_min, lim_max],
            "k--", lw=1.0, label="y = x")

    ax.set_xlabel("Actual $T_c$ (K)", fontsize=9)
    ax.set_ylabel("Predicted $T_c$ (K)", fontsize=9)
    ax.set_title(
        f"{fam}  (n={n})\n"
        f"R²={r2:.4f}  |  RMSE={rmse:.2f} K",
        fontsize=8.5,
    )
    ax.tick_params(labelsize=8)
    ax.grid(True, alpha=0.3)

# fig.suptitle(
#     f"Stacking Ensemble — Per-Family Performance on Held-Out Test Set\n"
#     f"Overall: R²={overall_r2:.4f}  |  RMSE={overall_rmse:.2f} K  "
#     f"|  MAE={overall_mae:.2f} K  |  n_test={len(y_test):,}",
#     fontsize=12,
#     fontweight="bold",
#     y=1.005,
# )
print(fig)


Figure(1800x1200)


In [32]:

plt.savefig("per_family_breakdown.png", dpi=600,
            bbox_inches="tight", facecolor="white")
plt.close()
print("\nSaved plot -> per_family_breakdown.png")



Saved plot -> per_family_breakdown.png


In [ ]:

# =============================================================================
# 8. SAVE METRICS TABLE AS CSV
# =============================================================================
summary_row = pd.DataFrame([{
    "family": "OVERALL",
    "n"     : len(y_test),
    "R2"    : round(overall_r2,   4),
    "RMSE"  : round(overall_rmse, 4),
    "MAE"   : round(overall_mae,  4),
}])

df_export = pd.concat(
    [df_results.round(4), summary_row],
    ignore_index=True
)


In [ ]:

df_export.to_csv("per_family_metrics.csv", index=False)
print("Saved table  -> per_family_metrics.csv")
print("\nDone.")